# B8. E-commerce Dashboard Data Processing Notebook

> **Related Module**: [B8 E-commerce Dashboard](../paths/b-developers/b8-ecommerce-dashboard.md)
>
> **Features**: Multi-platform data integration + KPI calculation + anomaly detection + visualization
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/b8-dashboard-demo.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q pandas numpy plotly

## 2. Generate Sample Data

Simulate 90 days of sales data across Amazon + Shopify dual platforms.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
days = 90
dates = pd.date_range(end=datetime.now(), periods=days, freq='D')

# Amazon data
amazon = pd.DataFrame({
    'date': dates,
    'platform': 'amazon',
    'revenue': np.random.normal(1500, 300, days).clip(500),
    'orders': np.random.poisson(45, days),
    'units': np.random.poisson(52, days),
    'ad_spend': np.random.normal(200, 50, days).clip(50),
    'ad_revenue': np.random.normal(800, 200, days).clip(100),
    'refunds': np.random.poisson(2, days) * 25,
    'cogs': np.random.normal(450, 80, days).clip(200),
    'fba_fees': np.random.normal(180, 30, days).clip(80)
})

# Shopify data
shopify = pd.DataFrame({
    'date': dates,
    'platform': 'shopify',
    'revenue': np.random.normal(600, 150, days).clip(100),
    'orders': np.random.poisson(15, days),
    'units': np.random.poisson(18, days),
    'ad_spend': np.random.normal(100, 30, days).clip(20),
    'ad_revenue': np.random.normal(350, 100, days).clip(50),
    'refunds': np.random.poisson(1, days) * 30,
    'cogs': np.random.normal(180, 40, days).clip(50),
    'fba_fees': np.zeros(days)  # No FBA for Shopify
})

# Inject anomaly on day 75
amazon.loc[74, 'revenue'] = 300  # Revenue drop
amazon.loc[74, 'ad_spend'] = 500  # Ad spend spike

df = pd.concat([amazon, shopify], ignore_index=True)
df['net_profit'] = df['revenue'] - df['cogs'] - df['fba_fees'] - df['ad_spend'] - df['refunds']
df['acos'] = (df['ad_spend'] / df['ad_revenue'].replace(0, 1)) * 100
df['roas'] = df['ad_revenue'] / df['ad_spend'].replace(0, 1)

print(f'Total records: {len(df)}')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Platforms: {df["platform"].unique()}')
df.head()

## 3. KPI Calculation

In [ ]:
# Overall KPIs
total = df.groupby('platform').agg({
    'revenue': 'sum',
    'orders': 'sum',
    'ad_spend': 'sum',
    'ad_revenue': 'sum',
    'net_profit': 'sum',
    'refunds': 'sum'
}).round(0)

total['aov'] = (total['revenue'] / total['orders']).round(2)
total['roas'] = (total['ad_revenue'] / total['ad_spend']).round(1)
total['acos'] = ((total['ad_spend'] / total['ad_revenue']) * 100).round(1)
total['margin'] = ((total['net_profit'] / total['revenue']) * 100).round(1)
total['refund_rate'] = ((total['refunds'] / total['revenue']) * 100).round(1)

print('=== Platform KPI Summary (90 days) ===')
display(total[['revenue', 'orders', 'aov', 'roas', 'acos', 'margin', 'refund_rate']])

print(f'\nTotal Revenue: ${total["revenue"].sum():,.0f}')
print(f'Total Profit: ${total["net_profit"].sum():,.0f}')
print(f'Overall Margin: {total["net_profit"].sum() / total["revenue"].sum() * 100:.1f}%')

## 4. Anomaly Detection

In [ ]:
def detect_anomalies(series, window=7, threshold=2.0):
    """Z-Score based anomaly detection"""
    rolling_mean = series.rolling(window=window).mean()
    rolling_std = series.rolling(window=window).std()
    z_scores = (series - rolling_mean) / rolling_std
    return z_scores, abs(z_scores) > threshold

# Check Amazon revenue anomalies
amz = df[df['platform'] == 'amazon'].copy().reset_index(drop=True)
z_scores, is_anomaly = detect_anomalies(amz['revenue'])

anomalies = amz[is_anomaly]
print(f'=== Anomalies Detected: {len(anomalies)} ===')
for _, row in anomalies.iterrows():
    direction = '📈 HIGH' if z_scores[row.name] > 0 else '📉 LOW'
    print(f'{row["date"].date()}: Revenue ${row["revenue"]:.0f} ({direction}, z={z_scores[row.name]:.1f})')

## 5. Visualization

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Daily revenue by platform
daily = df.groupby(['date', 'platform'])['revenue'].sum().reset_index()
fig = px.line(daily, x='date', y='revenue', color='platform',
              title='Daily Revenue by Platform')
fig.show()

# Profit waterfall (Amazon)
amz_total = df[df['platform'] == 'amazon'].sum()
waterfall = go.Figure(go.Waterfall(
    x=['Revenue', 'COGS', 'FBA Fees', 'Ad Spend', 'Refunds', 'Net Profit'],
    y=[amz_total['revenue'], -amz_total['cogs'], -amz_total['fba_fees'],
       -amz_total['ad_spend'], -amz_total['refunds'], amz_total['net_profit']],
    measure=['absolute', 'relative', 'relative', 'relative', 'relative', 'total'],
    connector={'line': {'color': 'rgb(63, 63, 63)'}}
))
waterfall.update_layout(title='Amazon Profit Waterfall (90 days)')
waterfall.show()

# ROAS trend
amz_daily = df[df['platform'] == 'amazon'].copy()
amz_daily['roas_7d'] = amz_daily['ad_revenue'].rolling(7).sum() / amz_daily['ad_spend'].rolling(7).sum()
fig = px.line(amz_daily, x='date', y='roas_7d', title='Amazon 7-Day Rolling ROAS')
fig.add_hline(y=3, line_dash='dash', line_color='red', annotation_text='Target ROAS=3x')
fig.show()

## 6. Export Report

In [ ]:
# Export
df.to_csv('dashboard_data.csv', index=False)
print('Data exported to dashboard_data.csv')
print('\nNext step: Use this data with Streamlit dashboard')
print('Run: streamlit run dashboard.py')
print('See: paths/b-developers/b8-ecommerce-dashboard.md for full code')